# 11c — Dipole positions on the LSSTCam Focal Plane

This notebook visualises **where dipole-flagged DIA sources fall on the
LSSTCam focal plane**, using the same geometry CSV as notebook 05.

**Scientific motivation:**  
Dipole artefacts in difference images arise when a source moves between the
science exposure and the template coadd (proper motion, astrometric error)
or when the PSF-matching kernel is imperfect.  A non-uniform distribution of
dipoles across the focal plane could reveal:
- CCD-level PSF-modelling issues (e.g. edge sensors, corner rafts),
- detector / readout artefacts concentrated on specific chips,
- focal-plane illumination gradients that degrade template quality at the edges.

## Figures produced

| Figure | Description |
|--------|-------------|
| **Figure 1** | All-band scatter: dipole alert positions on the focal plane, colour-coded by Gaia category. |
| **Figure 2** | 2×3 per-band scatter: one subplot per band (`u g r i z y`), same colour coding by category. |
| **Figure 3** | All-band heatmap: dipole count per CCD (log colour scale) with CCD annotations. |
| **Figure 4** | 2×3 per-band heatmaps: shared log colour scale across all bands. |
| **Figure 5** | Dipole fraction per CCD: N_dipole / N_total, all-band heatmap. |
| **Figure 6** | Radial analysis: dipole fraction vs focal-plane radius. |

**Stellar categories:**
- `gaia_star_stable_hq` — Gaia stable HQ (blue)
- `gaia_nophotgstar_stable_unknown_parallax` — Gaia stable, no photometric solution (green)
- `gaia_star_variable` — Gaia variable stars (red)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS, Université Paris-Saclay)  
**Creation Date:** 2026-05-18  
**Notebook tag:** 11c  
**Requires:** `data_FINK_BLOCK_LC_01/{cat}_src.parquet`, `data_FocalPlane/ccd_geometry.csv`

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

print(f"pandas    : {pd.__version__}")
print(f"numpy     : {np.__version__}")
print(f"matplotlib: {mpl.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl not found → inline backend")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: parquets from notebook 01
DIR_FIGS = "figs_FINK_BLOCK_LC_11c"  # output: figures specific to this notebook
GEOMETRY_CSV = "data_FocalPlane/ccd_geometry.csv"
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BANDS_ROW0 = list("ugr")
BANDS_ROW1 = list("izy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Stellar categories — colour per category for scatter plots ─────────────────
CATEGORIES = {
    "gaia_star_stable_hq": {
        "label": "Gaia stable HQ",
        "color": "steelblue",
        "marker": "o",
    },
    "gaia_nophotgstar_stable_unknown_parallax": {
        "label": "Gaia stable no-phot",
        "color": "seagreen",
        "marker": "s",
    },
    "gaia_star_variable": {
        "label": "Gaia variable",
        "color": "firebrick",
        "marker": "^",
    },
}

# ── Marker sizes & transparency ────────────────────────────────────────────────
MS_SCATTER = 6  # marker size for scatter on focal plane
ALPHA_SCATTER = 0.55

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": False,  # grid off for focal-plane maps
        "font.size": 9,
    }
)


def savefig(name):
    """Save the current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data dir      : {os.path.abspath(DIR_DATA)}")
print(f"Figs dir      : {os.path.abspath(DIR_FIGS)}")
print(f"Geometry CSV  : {os.path.abspath(GEOMETRY_CSV)}")

## 2. Utility functions

In [ ]:
def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Coerce a dipole-flag column (bool / int / str) to a boolean Series."""

    def _cast(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


def draw_focal_plane_outline(
    ax,
    geom: pd.DataFrame,
    facecolor="white",
    edgecolor="#555555",
    linewidth=0.4,
    annotate_detector=True,
    annotate_name=False,
    fontsize_det=4,
    fontsize_name=3,
):
    """
    Draw the LSSTCam focal plane outline as a collection of CCD polygons.

    Parameters
    ----------
    ax               : matplotlib axes
    geom             : geometry DataFrame from ccd_geometry.csv
    facecolor        : fill colour for each CCD polygon
    edgecolor        : edge colour for CCD borders
    linewidth        : border line width
    annotate_detector: if True, print the detector number at the CCD centre
    annotate_name    : if True, also print the CCD name below the number
    fontsize_det     : font size for detector number annotation
    fontsize_name    : font size for CCD name annotation
    """
    for _, row in geom.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        poly = patches.Polygon(
            corners,
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=linewidth,
            zorder=1,
        )
        ax.add_patch(poly)

        if annotate_detector:
            label = str(int(row["detector"]))
            if annotate_name and "name" in row.index and pd.notna(row["name"]):
                label += f"\n{row['name']}"
            ax.text(
                row["x_center"],
                row["y_center"],
                label,
                ha="center",
                va="center",
                fontsize=fontsize_det if not annotate_name else fontsize_name,
                fontweight="bold",
                color="#333333",
                zorder=2,
            )


def get_fp_bbox(geom: pd.DataFrame):
    """Return (xmin, xmax, ymin, ymax) of the focal plane bounding box."""
    xmin = geom[[f"corner{i}_x" for i in range(4)]].min().min()
    xmax = geom[[f"corner{i}_x" for i in range(4)]].max().max()
    ymin = geom[[f"corner{i}_y" for i in range(4)]].min().min()
    ymax = geom[[f"corner{i}_y" for i in range(4)]].max().max()
    return xmin, xmax, ymin, ymax


print("Utility functions defined.")

## 3. Load focal-plane geometry

In [ ]:
geom = pd.read_csv(GEOMETRY_CSV)

# Compute radial distance from focal-plane centre for each CCD
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

fp_xmin, fp_xmax, fp_ymin, fp_ymax = get_fp_bbox(geom)

print(f"Geometry CSV : {GEOMETRY_CSV}")
print(f"Total CCDs   : {len(geom)}")
print(f"Columns      : {list(geom.columns)}")
print(f"FP bbox      : X=[{fp_xmin:.1f}, {fp_xmax:.1f}]  Y=[{fp_ymin:.1f}, {fp_ymax:.1f}]")
display(geom.head(4))

## 4. Load parquet data and extract dipole alerts

Three stellar categories are loaded from the `data_FINK_BLOCK_LC_01/` parquets
(same files as notebook 11b).  Only rows with `r:isDipole = True` and a valid
focal-plane detector coordinate are retained for the focal-plane scatter.

The column `r:detector` holds the LSSTCam detector number (integer 0–188),
which is joined to the geometry CSV to obtain the focal-plane position of each
alert.  The per-alert x/y position **within** a CCD is not stored in the
alert; we therefore use the CCD **centre coordinates** as the scatter position
and add a small uniform random jitter (± half CCD width) to separate overlapping
points from the same detector.

> **Note on jitter:**  The jitter is seeded deterministically so the figure is
> reproducible.  Set `JITTER_FRACTION = 0` to disable it and plot all points
> exactly at the CCD centre.

In [ ]:
# ── Jitter parameters ──────────────────────────────────────────────────────────
# Fraction of the CCD half-width used as the jitter amplitude.
# 0 → all points at the CCD centre (fully stacked)
# 1 → points spread uniformly across the full CCD area
JITTER_FRACTION = 0.85
RNG_SEED = 42

# ── Estimate CCD half-size from geometry ───────────────────────────────────────
ccd_dx = (
    geom[["corner0_x", "corner1_x", "corner2_x", "corner3_x"]].max(axis=1)
    - geom[["corner0_x", "corner1_x", "corner2_x", "corner3_x"]].min(axis=1)
) / 2.0
ccd_dy = (
    geom[["corner0_y", "corner1_y", "corner2_y", "corner3_y"]].max(axis=1)
    - geom[["corner0_y", "corner1_y", "corner2_y", "corner3_y"]].min(axis=1)
) / 2.0

# Build a lookup: detector → (x_center, y_center, half_dx, half_dy)
det_geom = geom.set_index("detector")[["x_center", "y_center"]].copy()
det_geom["half_dx"] = ccd_dx.values
det_geom["half_dy"] = ccd_dy.values

print(f"Median CCD half-width  : {ccd_dx.median():.2f} mm")
print(f"Median CCD half-height : {ccd_dy.median():.2f} mm")

In [ ]:
rng = np.random.default_rng(RNG_SEED)

frames_all = []  # all alerts (dipole + non-dipole) for fraction plot
frames_dip = []  # dipole-only alerts

for cat, cat_meta in CATEGORIES.items():
    fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: file not found ({fpath})")
        continue

    df = pd.read_parquet(fpath)

    # ── Dipole flag ────────────────────────────────────────────────────────
    if "r:isDipole" in df.columns:
        df["is_dipole"] = parse_dipole_bool(df["r:isDipole"].fillna(False))
    else:
        print(f"[WARN] {cat}: no r:isDipole column — all set to False")
        df["is_dipole"] = False

    # ── Check detector column ──────────────────────────────────────────────
    if "r:detector" not in df.columns:
        print(f"[SKIP] {cat}: no r:detector column")
        continue

    # ── Keep only rows with a valid detector ───────────────────────────────
    df = df.dropna(subset=["r:detector"]).copy()
    df["detector"] = df["r:detector"].astype(int)

    # ── Map to focal-plane coordinates + jitter ────────────────────────────
    valid_det = df["detector"].isin(det_geom.index)
    df = df[valid_det].copy().reset_index(drop=True)

    xc = det_geom.loc[df["detector"], "x_center"].values
    yc = det_geom.loc[df["detector"], "y_center"].values
    hdx = det_geom.loc[df["detector"], "half_dx"].values
    hdy = det_geom.loc[df["detector"], "half_dy"].values

    n = len(df)
    jx = rng.uniform(-JITTER_FRACTION, JITTER_FRACTION, size=n) * hdx
    jy = rng.uniform(-JITTER_FRACTION, JITTER_FRACTION, size=n) * hdy

    df["fp_x"] = xc + jx
    df["fp_y"] = yc + jy
    df["gaia_origin"] = cat
    df["cat_label"] = cat_meta["label"]
    df["cat_color"] = cat_meta["color"]
    df["cat_marker"] = cat_meta["marker"]

    n_dip = int(df["is_dipole"].sum())
    print(
        f"  {cat_meta['label']:40s}  {n:6d} total  {n_dip:5d} dipoles "
        f"({100 * n_dip / n:.1f}%)  [{df['r:band'].nunique()} bands]"
    )

    frames_all.append(df)
    frames_dip.append(df[df["is_dipole"]].copy())

if not frames_all:
    raise RuntimeError("No parquet files loaded — check DIR_DATA path.")

df_all = pd.concat(frames_all, ignore_index=True)
df_dip = pd.concat(frames_dip, ignore_index=True)

print(f"\nAll alerts (all categories, all bands): {len(df_all):,}")
print(f"Dipole alerts                          : {len(df_dip):,}")

print("\nDipole counts per band:")
display(df_dip.groupby("r:band")["gaia_origin"].count().rename("N_dipoles").reset_index())

## 5. Figure 1 — All-band dipole scatter on focal plane

All dipole-flagged alerts are scattered at their (jittered) CCD position
on the focal plane.  Marker colour encodes the Gaia category.

CCD outlines are drawn in light grey.  Each CCD is annotated with its
detector number and name.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(9, 9))

# ── Focal-plane outline ────────────────────────────────────────────────────
draw_focal_plane_outline(
    ax1,
    geom,
    facecolor="#f5f5f5",
    edgecolor="#999999",
    linewidth=0.4,
    annotate_detector=True,
    annotate_name=True,
    fontsize_det=4,
    fontsize_name=3,
)

# ── Dipole scatter — one layer per category ────────────────────────────────
for cat, cat_meta in CATEGORIES.items():
    sub = df_dip[df_dip["gaia_origin"] == cat]
    if sub.empty:
        continue
    ax1.scatter(
        sub["fp_x"],
        sub["fp_y"],
        s=MS_SCATTER,
        c=cat_meta["color"],
        marker=cat_meta["marker"],
        alpha=ALPHA_SCATTER,
        linewidths=0,
        zorder=5,
        label=f"{cat_meta['label']} (N={len(sub):,})",
    )

ax1.set_aspect("equal")
ax1.set_xlim(fp_xmin * 1.02, fp_xmax * 1.02)
ax1.set_ylim(fp_ymin * 1.02, fp_ymax * 1.02)
ax1.set_xlabel("Focal plane X  [mm]", fontsize=10)
ax1.set_ylabel("Focal plane Y  [mm]", fontsize=10)
ax1.set_title(
    f"Dipole positions on LSSTCam focal plane — all bands combined\n(N_dipole = {len(df_dip):,})",
    fontsize=11,
    fontweight="bold",
)
ax1.legend(fontsize=8, loc="upper right", framealpha=0.85, markerscale=2)

plt.tight_layout()
savefig("11c_fig1_dipoles_allbands_scatter")
plt.show()

## 6. Figure 2 — Per-band dipole scatter (2×3 grid)

Same as Figure 1 but split by photometric band.  Each subplot shows only the
dipole-flagged alerts from that band; the colour still encodes the Gaia category.

Layout: u g r (top row), i z y (bottom row).

In [ ]:
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)

fig2.suptitle(
    "Dipole positions on LSSTCam focal plane — per band\n"
    "colour = Gaia category   ● stable HQ   ■ stable no-phot   ▲ variable",
    fontsize=12,
    fontweight="bold",
)

for idx, band in enumerate(BANDS):
    row_idx, col_idx = divmod(idx, 3)
    ax = axes2[row_idx, col_idx]
    bcolor = BAND_COLORS[band]

    # ── Focal-plane outline ────────────────────────────────────────────────
    draw_focal_plane_outline(
        ax,
        geom,
        facecolor="#f8f8f8",
        edgecolor="#bbbbbb",
        linewidth=0.3,
        annotate_detector=True,
        annotate_name=False,
        fontsize_det=3,
    )

    df_band = df_dip[df_dip["r:band"] == band]
    n_band_dip = len(df_band)

    # ── Dipole scatter — one layer per category ────────────────────────────
    for cat, cat_meta in CATEGORIES.items():
        sub = df_band[df_band["gaia_origin"] == cat]
        if sub.empty:
            continue
        ax.scatter(
            sub["fp_x"],
            sub["fp_y"],
            s=MS_SCATTER,
            c=cat_meta["color"],
            marker=cat_meta["marker"],
            alpha=ALPHA_SCATTER,
            linewidths=0,
            zorder=5,
            label=f"{cat_meta['label']} (N={len(sub):,})",
        )

    ax.set_aspect("equal")
    ax.set_xlim(fp_xmin * 1.02, fp_xmax * 1.02)
    ax.set_ylim(fp_ymin * 1.02, fp_ymax * 1.02)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(
        rf"Band {band}   (N_dipole = {n_band_dip:,})",
        fontsize=11,
        fontweight="bold",
        color=bcolor,
    )
    if n_band_dip > 0:
        ax.legend(fontsize=6, loc="upper right", framealpha=0.85, markerscale=2)

savefig("11c_fig2_dipoles_perband_scatter")
plt.show()

## 7. Figure 3 — All-band dipole count heatmap per CCD

Each CCD is filled with a colour proportional to the **number of dipole alerts**
falling in that CCD (log scale).  CCDs with zero dipole alerts are shown in light
grey.  Detector numbers and CCD names are annotated.

In [ ]:
# ── Per-CCD dipole counts — all bands ─────────────────────────────────────────
dip_counts_all = df_dip.groupby("detector")["is_dipole"].count().rename("n_dip").reset_index()
geom_dip = geom.merge(dip_counts_all, on="detector", how="left")
geom_dip["n_dip"] = geom_dip["n_dip"].fillna(0).astype(float)

# Colour normalisation (log)
n_dip_nonzero = geom_dip["n_dip"][geom_dip["n_dip"] > 0]
vmin_dip = float(n_dip_nonzero.min()) if len(n_dip_nonzero) else 1.0
vmax_dip = float(geom_dip["n_dip"].max())
norm_dip_all = mcolors.LogNorm(vmin=max(vmin_dip, 0.5), vmax=vmax_dip)
cmap_dip = plt.get_cmap("plasma")

print(f"All-band dipole counts per CCD — range: [{vmin_dip:.0f}, {vmax_dip:.0f}]")

In [ ]:
fig3, ax3 = plt.subplots(figsize=(9, 9))

for _, row in geom_dip.iterrows():
    corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
    val = row["n_dip"]
    facecolor = cmap_dip(norm_dip_all(val)) if val > 0 else "#eeeeee"
    poly = patches.Polygon(
        corners,
        facecolor=facecolor,
        edgecolor="black",
        linewidth=0.25,
        zorder=1,
    )
    ax3.add_patch(poly)

    # Annotate: detector number + CCD name (if available)
    label = str(int(row["detector"]))
    if "name" in row.index and pd.notna(row["name"]):
        label += f"\n{row['name']}"
    ax3.text(
        row["x_center"],
        row["y_center"],
        label,
        ha="center",
        va="center",
        fontsize=4,
        fontweight="bold",
        color="#222222",
        zorder=2,
    )

ax3.set_aspect("equal")
ax3.set_xlim(fp_xmin * 1.02, fp_xmax * 1.02)
ax3.set_ylim(fp_ymin * 1.02, fp_ymax * 1.02)
ax3.set_xlabel("Focal plane X  [mm]", fontsize=10)
ax3.set_ylabel("Focal plane Y  [mm]", fontsize=10)
ax3.set_title(
    f"Dipole count per CCD — all bands (log scale)\nN_dipole total = {len(df_dip):,}",
    fontsize=11,
    fontweight="bold",
)

sm3 = plt.cm.ScalarMappable(cmap=cmap_dip, norm=norm_dip_all)
sm3.set_array([])
cbar3 = fig3.colorbar(sm3, ax=ax3, fraction=0.046, pad=0.04)
cbar3.set_label("Dipole count per CCD  (log scale)", fontsize=9)

plt.tight_layout()
savefig("11c_fig3_dipole_heatmap_allbands")
plt.show()

## 8. Figure 4 — Per-band dipole count heatmaps (2×3 grid)

Six heatmaps with a **shared log colour scale** so that inter-band differences
in dipole rate are visually comparable.  CCDs without dipoles in a given band
are shown in light grey.

In [ ]:
# ── Per-band dipole counts ─────────────────────────────────────────────────────
band_dip_counts = {}  # band → Series indexed by detector
for band in BANDS:
    df_b = df_dip[df_dip["r:band"] == band]
    if len(df_b) == 0:
        band_dip_counts[band] = pd.Series(dtype=float)
    else:
        band_dip_counts[band] = df_b.groupby("detector")["is_dipole"].count()

# ── Global colour scale (shared across bands) ──────────────────────────────────
all_nonzero = pd.concat([s for s in band_dip_counts.values() if len(s) > 0])
g_vmin = float(all_nonzero[all_nonzero > 0].min()) if (all_nonzero > 0).any() else 1.0
g_vmax = float(all_nonzero.max())
norm_shared = mcolors.LogNorm(vmin=max(g_vmin, 0.5), vmax=g_vmax)
cmap_shared = plt.get_cmap("plasma")

print(f"Global dipole count range (all bands): [{g_vmin:.0f}, {g_vmax:.0f}]")
for band in BANDS:
    s = band_dip_counts[band]
    n = int(s.sum()) if len(s) else 0
    print(f"  band {band}: {n:5,} dipoles  |  {int((s > 0).sum()) if len(s) else 0} CCDs with dipoles")

In [ ]:
fig4, axes4 = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)

for idx, band in enumerate(BANDS):
    row_idx, col_idx = divmod(idx, 3)
    ax = axes4[row_idx, col_idx]
    bcolor = BAND_COLORS[band]

    # Map dipole counts to geometry for this band
    geom_b = geom.copy()
    geom_b["n_dip"] = geom_b["detector"].map(band_dip_counts[band]).fillna(0).astype(float)

    for _, row in geom_b.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row["n_dip"]
        facecolor = cmap_shared(norm_shared(val)) if val > 0 else "#eeeeee"
        poly = patches.Polygon(
            corners,
            facecolor=facecolor,
            edgecolor="black",
            linewidth=0.2,
            zorder=1,
        )
        ax.add_patch(poly)
        # Detector number only (space is tight in 2x3 grid)
        ax.text(
            row["x_center"],
            row["y_center"],
            str(int(row["detector"])),
            ha="center",
            va="center",
            fontsize=3,
            fontweight="bold",
            color="#333333",
            zorder=2,
        )

    ax.set_aspect("equal")
    ax.set_xlim(fp_xmin * 1.02, fp_xmax * 1.02)
    ax.set_ylim(fp_ymin * 1.02, fp_ymax * 1.02)
    ax.set_xticks([])
    ax.set_yticks([])
    n_band = int(band_dip_counts[band].sum()) if len(band_dip_counts[band]) else 0
    ax.set_title(
        rf"Band {band}   (N_dipole = {n_band:,})",
        fontsize=11,
        fontweight="bold",
        color=bcolor,
    )

# Single shared colour bar
sm4 = plt.cm.ScalarMappable(cmap=cmap_shared, norm=norm_shared)
sm4.set_array([])
cbar4 = fig4.colorbar(sm4, ax=axes4, fraction=0.02, pad=0.02)
cbar4.set_label("Dipole count per CCD  (log scale, shared across bands)", fontsize=11)

fig4.suptitle(
    "LSSTCam focal plane — Dipole count per band\n"
    "(shared log colour scale  |  grey = no dipole in this band)",
    fontsize=13,
)

savefig("11c_fig4_dipole_heatmap_perband")
plt.show()

## 9. Figure 5 — Dipole fraction per CCD (all bands)

The **dipole fraction** = N_dipole / N_total per CCD reveals which CCDs
preferentially produce dipole-flagged alerts **independently of the
alert rate**.  A high fraction indicates a systematic issue localised
to that chip (PSF model, template quality, read-noise pattern…).

In [ ]:
# ── Per-CCD total and dipole counts ───────────────────────────────────────────
total_counts = df_all.groupby("detector")["is_dipole"].count().rename("n_total").reset_index()
dipole_counts = df_all.groupby("detector")["is_dipole"].sum().rename("n_dip").reset_index()
geom_frac = geom.merge(total_counts, on="detector", how="left")
geom_frac = geom_frac.merge(dipole_counts, on="detector", how="left")
geom_frac["n_total"] = geom_frac["n_total"].fillna(0).astype(float)
geom_frac["n_dip"] = geom_frac["n_dip"].fillna(0).astype(float)
geom_frac["frac_dip"] = np.where(
    geom_frac["n_total"] > 0,
    geom_frac["n_dip"] / geom_frac["n_total"],
    np.nan,
)

frac_finite = geom_frac["frac_dip"].dropna()
print(
    f"Dipole fraction across CCDs — median: {frac_finite.median():.4f}  "
    f"max: {frac_finite.max():.4f}  min: {frac_finite.min():.4f}"
)
display(
    geom_frac[["detector", "n_total", "n_dip", "frac_dip"]].sort_values("frac_dip", ascending=False).head(15)
)

In [ ]:
# ── Colour scale for fraction ──────────────────────────────────────────────────
frac_vmin = 0.0
frac_vmax = float(np.nanpercentile(geom_frac["frac_dip"].values, 99))
norm_frac = mcolors.Normalize(vmin=frac_vmin, vmax=frac_vmax)
cmap_frac = plt.get_cmap("hot_r")

fig5, ax5 = plt.subplots(figsize=(9, 9))

for _, row in geom_frac.iterrows():
    corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
    val = row["frac_dip"]
    if np.isnan(val):
        facecolor = "#dddddd"  # no alerts on this CCD at all
    elif val == 0:
        facecolor = "#f0f0f0"  # alerts but no dipoles
    else:
        facecolor = cmap_frac(norm_frac(val))
    poly = patches.Polygon(
        corners,
        facecolor=facecolor,
        edgecolor="black",
        linewidth=0.25,
        zorder=1,
    )
    ax5.add_patch(poly)

    # Annotation: detector number + fraction value (if non-zero)
    label = str(int(row["detector"]))
    if "name" in row.index and pd.notna(row["name"]):
        label += f"\n{row['name']}"
    if np.isfinite(val) and val > 0:
        label += f"\n{val:.2f}"
    ax5.text(
        row["x_center"],
        row["y_center"],
        label,
        ha="center",
        va="center",
        fontsize=3.5,
        color="#222222",
        fontweight="bold",
        zorder=2,
    )

ax5.set_aspect("equal")
ax5.set_xlim(fp_xmin * 1.02, fp_xmax * 1.02)
ax5.set_ylim(fp_ymin * 1.02, fp_ymax * 1.02)
ax5.set_xlabel("Focal plane X  [mm]", fontsize=10)
ax5.set_ylabel("Focal plane Y  [mm]", fontsize=10)
ax5.set_title(
    "Dipole fraction per CCD — all bands  (N_dipole / N_total)\n"
    "dark grey = no alerts  |  light grey = alerts but 0 dipoles  |  colour = fraction",
    fontsize=11,
    fontweight="bold",
)

sm5 = plt.cm.ScalarMappable(cmap=cmap_frac, norm=norm_frac)
sm5.set_array([])
cbar5 = fig5.colorbar(sm5, ax=ax5, fraction=0.046, pad=0.04)
cbar5.set_label("Dipole fraction  N_dipole / N_total", fontsize=9)

plt.tight_layout()
savefig("11c_fig5_dipole_fraction_allbands")
plt.show()

## 10. Summary statistics per CCD

In [ ]:
cols_show = ["detector", "n_total", "n_dip", "frac_dip", "x_center", "y_center", "r_mm"]
if "name" in geom_frac.columns:
    cols_show.insert(1, "name")

df_ccd_summary = geom_frac[cols_show].copy()

print("Top 20 CCDs by DIPOLE COUNT (all bands):")
display(
    df_ccd_summary.sort_values("n_dip", ascending=False)
    .head(20)
    .reset_index(drop=True)
    .style.format({"frac_dip": "{:.4f}", "r_mm": "{:.1f}"})
)

print("\nTop 20 CCDs by DIPOLE FRACTION (all bands, min 10 total alerts):")
display(
    df_ccd_summary[df_ccd_summary["n_total"] >= 10]
    .sort_values("frac_dip", ascending=False)
    .head(20)
    .reset_index(drop=True)
    .style.format({"frac_dip": "{:.4f}", "r_mm": "{:.1f}"})
)

## 11. Figure 6 — Radial analysis: dipole fraction vs focal-plane radius

To check whether dipoles are preferentially produced at the edge of the focal
plane (where PSF models are harder to constrain and template quality may be
lower), we plot the **per-CCD dipole fraction** vs radial distance from the
focal-plane centre.

In [ ]:
df_radial = geom_frac[geom_frac["n_total"] >= 5].copy()

fig6, axes6 = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# ── Left panel: fraction vs radius ────────────────────────────────────────────
sc = axes6[0].scatter(
    df_radial["r_mm"],
    df_radial["frac_dip"],
    c=df_radial["n_dip"],
    cmap="plasma",
    norm=mcolors.LogNorm(
        vmin=max(df_radial["n_dip"][df_radial["n_dip"] > 0].min(), 0.5),
        vmax=df_radial["n_dip"].max(),
    ),
    s=40,
    edgecolors="grey",
    linewidths=0.4,
    zorder=3,
)
axes6[0].set_xlabel(r"Radial distance  $r = \sqrt{x^2+y^2}$  [mm]", fontsize=10)
axes6[0].set_ylabel("Dipole fraction  N_dipole / N_total", fontsize=10)
axes6[0].set_title(
    "Dipole fraction vs focal-plane radius\ncolour = N_dipole per CCD  (log)",
    fontsize=10,
)
axes6[0].grid(True, alpha=0.25)
cb = fig6.colorbar(sc, ax=axes6[0], fraction=0.046, pad=0.04)
cb.set_label("N_dipole / CCD (log)", fontsize=8)

# ── Right panel: N_dipole vs N_total per CCD ──────────────────────────────────
sc2 = axes6[1].scatter(
    df_radial["n_total"],
    df_radial["n_dip"],
    c=df_radial["r_mm"],
    cmap="viridis",
    s=35,
    edgecolors="grey",
    linewidths=0.4,
    alpha=0.8,
    zorder=3,
)
med_frac = float(df_radial["frac_dip"].median())
xmax_ref = df_radial["n_total"].max() * 1.05
x_ref = np.linspace(0, xmax_ref, 200)
axes6[1].plot(x_ref, med_frac * x_ref, "k--", lw=1.0, label=f"median fraction = {med_frac:.4f}")
axes6[1].set_xlabel("N_total per CCD", fontsize=10)
axes6[1].set_ylabel("N_dipole per CCD", fontsize=10)
axes6[1].set_title(
    "N_dipole vs N_total per CCD\ncolour = radial distance r [mm]",
    fontsize=10,
)
axes6[1].grid(True, alpha=0.25)
axes6[1].legend(fontsize=8)
cb2 = fig6.colorbar(sc2, ax=axes6[1], fraction=0.046, pad=0.04)
cb2.set_label("r  [mm]", fontsize=8)

fig6.suptitle(
    "Radial analysis of dipole fraction on LSSTCam focal plane (all bands)",
    fontsize=12,
    fontweight="bold",
)

savefig("11c_fig6_dipole_fraction_vs_radius")
plt.show()

## 12. Conclusions

**Figures 1 & 2 (scatter):**
- The spatial distribution of dipole alerts across the focal plane reveals
  whether dipoles are concentrated on specific CCDs or distributed uniformly.
- If dipoles cluster on a subset of chips, those CCDs likely have PSF-modelling
  or template-building issues.
- The colour coding by Gaia category (stable HQ / stable no-phot / variable)
  shows whether the dipole pattern is category-dependent: if variable stars
  (red) are more uniformly distributed while stable stars (blue/green) cluster
  on specific CCDs, the CCD-level explanation is favoured over an astrophysical
  explanation.

**Figures 3 & 4 (heatmaps):**
- A non-uniform dipole count with hot spots at the focal-plane edges points to
  vignetting or field-curvature-induced PSF degradation.
- Band-dependent hot spots (Figure 4) would indicate filter-specific chromatic
  PSF effects.

**Figure 5 (dipole fraction):**
- CCDs with a high fraction but moderate count are particularly suspicious:
  they may have systematic calibration issues that inflate the dipole flag rate
  regardless of the photon statistics.

**Figure 6 (radial analysis):**
- A rising dipole fraction with radial distance would confirm an edge-of-field
  origin and could be used to define a quality cut on detector position.

> **Recommended next steps:**
> - Cross-check the hot-spot CCDs against AP pipeline QA metrics
>   (PSF χ², template depth, sky background level).
> - Fetch Butler difference-image stamps for the top-fraction CCDs
>   to inspect the dipole morphology directly (notebook 12b).
> - Compare the focal-plane dipole map with the single-visit PSF FWHM
>   map from the Butler `calexp` summary catalogue.

In [ ]:
print("Notebook 11c — Dipole positions on LSSTCam focal plane — complete.")